# Projeto: Inserção dos dados na camada Gold do banco de dados.

### Descrição Geral

Este documento descreve o processo de cruzamento e enriquecimento de dados meteorológicos com informações críticas de limite pluviométrico municipal, visando a construção de uma **base confiável e analiticamente robusta** para a identificação de **possíveis eventos de alagamento**.

As fontes utilizadas são:
- A tabela `clima`, já presente na camada **gold**, contendo previsões climáticas padronizadas por município;
- O arquivo `chuvas_criticas`, oriundo da camada **silver**, previamente validado via **Great Expectations (GX)**, que contém o **volume máximo de chuva suportado** por cada localidade.

Neste estágio, o dado de **volume máximo de chuva (`vol_max_chuva`)** é adicionado ao dataset consolidado, possibilitando análises comparativas com a **precipitação prevista (`precipitacao`)**. Esta junção cria a fundação necessária para que, em processos posteriores, seja possível determinar com alta confiança os municípios em **potencial risco de alagamento**.

O resultado é persistido na tabela `clima_alagamento`, dentro da camada **gold**, servindo como **base estratégica para modelos de alerta, dashboards operacionais e ações preventivas.**


### Informações Técnicas

- **Fonte de Dados**:
  * Tabelas da base gold referentes a previsões climática
    * `clima`
  * Tabela Silver de Chuvas Críticas
    * `silver_chuvas_criticas`
- **Tecnologias Utilizadas**: 
  - Apache Spark (PySpark)
  - Databricks Notebooks

- **Dados persistidos**:
  - Data de carga
  - Nome do município
  - Sigla da unidade federativa (UF)
  - Data da carga dos dados
  - Data da previsão do clima
  - Precipitação prevista (em milímetros)
  - Probabilidade de precipitação prevista (em    porcentagem)
  - Volume máximo de chuva
  - Fonte da previsão do tempo
  - País
---

### Histórico de Atualizações

| Data       | Responsável      | Versão | Descrição                                |
|------------|------------------|--------|--------------------------------------------|
| 20/05/2025 | Lucas Bringmann      |  1.0   | Criação da inserção dos dados para camada gold|
| 20/05/2025 | Gabriel Rost      |  1.1   | Inclusão de documentação sobre enrequecimentos realizados|
| 21/05/2025 | Lucas Berta      |  1.2   | Alteração da documentação e union das tabelas para inserção|
| 21/05/2025 | Lucas Bringmann   |  1.3   | Ajuste de colunas para inserção do ID|
| 25/05/2025 | Evelise Dias      |  1.2  | 	Adição da leitura de arquivos validados pelo Great Expectations|
| 26/05/2025 | Evelise Dias      |  1.3  | 	Atualização da descrição geral do projeto e alinhamento com a arquitetura atual|

In [0]:
%run ../utils/utils_run

In [0]:
%run ./queries_gold

In [0]:
pasta_validos = lakehouse_logs_gx + "/silver/chuvas_criticas/validos/chuvas_criticas"
df_chuva = ler_tabelas(pasta_validos, fromTable=False)

In [0]:
df_clima = ler_tabelas(caminho_tabela_clima_gold)

In [0]:
df = (
    df_clima.alias("clima")
    .join(df_chuva.alias("chuva"), on="nom_municipio", how="inner")
    .select(
        col("chuva.vol_max_chuva"),
        col("clima.id"),
        col("clima.nom_municipio"),
        col("clima.nom_estado"),
        col("clima.dt_carga"),
        col("clima.dt_previsao"),
        col("clima.precipitacao").alias("precipitacao_prevista"),
        col("clima.prob_precipitacao").alias("prob_precipitacao_prevista"),
        col("clima.flg_fonte_previsao"),
        col("chuva.pais")
    )
)

In [0]:
spark.sql(create_table_gold_clima_alagamento)

In [0]:
inserir_com_merge(df, caminho_tabela_clima_alagamento_gold, ["id"])